# Financial News Sentiment Benchmarking Hub

This notebook implements and benchmarks three model architectures on the 3-class financial sentiment classification task: 
1. **Simple RNN** (Word Embeddings -> RNN -> Mean Pooling -> Dense Classifier)
2. **LSTM** (Word Embeddings -> LSTM -> Mean Pooling -> Dense Classifier)
3. **FinBERT** (Fine-tuning the pre-trained `ProsusAI/finbert` transformer)

---

## 1. Google Colab & Environment Setup

If you are running in Google Colab, mount Google Drive to save checkpoints and install dependencies.

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone repository or navigate to workspace directory on Drive
    # Replace with your actual directory path on Google Drive if needed
    # %cd /content/drive/MyDrive/financial-news-sentiment
    
    # Install dependencies
    print("Installing dependencies...")
    !pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn pyyaml streamlit
else:
    print("Running in Local environment.")

### Path Configuration

Ensure `src` package is in Python search path.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
print("Search path updated. Current working directory:", os.getcwd())

## 2. Phase 1 — Dataset & EDA

We load the `zeroshot/twitter-financial-news-sentiment` dataset, split sizes, analyze class distribution, compute tweet lengths, and check for leakage overlap.

In [ ]:
from src.phase1_eda import run_eda
run_eda()

## 3. Phase 2 — Preprocessing

The preprocessing module cleans URLs, user mentions, preserves tickers (e.g., `$AAPL`), and constructs the vocabulary exclusively from training data.

In [ ]:
from src.preprocessing import clean_text, tokenize, Vocabulary

sample_text = "Check out $AAPL and $TSLA! High gains today at http://example.com @user."
print("Original: ", sample_text)
print("Cleaned:  ", clean_text(sample_text))
print("Tokens:   ", tokenize(clean_text(sample_text)))

## 4. Phase 3 — Simple RNN Baseline

Train a 2-layer simple RNN classifier on CPU/GPU.

In [ ]:
!python ../src/training/train_rnn_lstm.py --model rnn

## 5. Phase 4 — LSTM Baseline

Train a 2-layer LSTM classifier on CPU/GPU.

In [ ]:
!python ../src/training/train_rnn_lstm.py --model lstm

## 6. Phase 5 — FinBERT Fine-Tuning (Colab GPU Recommended)

Fine-tune pre-trained `ProsusAI/finbert` using Hugging Face's `Trainer` API.

In [ ]:
# If running locally on CPU, this might take a very long time.
# If on Google Colab T4 GPU, it runs in ~2-3 minutes.
!python ../src/training/train_finbert.py

## 7. Phase 7 — Model Comparison

Compare validation metrics and plot accuracies/F1s.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

log_path = "../reports/experiment_log.csv"
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    print("--- Experiment Benchmarks ---")
    display(df)
    
    # Visualize Accuracy and F1
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sns.barplot(data=df, x="model", y="val_accuracy", palette="viridis")
    plt.title("Validation Accuracy Comparison")
    plt.ylim(0, 1.0)
    
    plt.subplot(1, 2, 2)
    sns.barplot(data=df, x="model", y="val_macro_f1", palette="viridis")
    plt.title("Validation Macro F1 Comparison")
    plt.ylim(0, 1.0)
    
    plt.tight_layout()
    plt.savefig("../figures/model_comparison.png")
    plt.show()
else:
    print("No log file found. Train the models first!")

## 8. View Confusion Matrices

Compare visual class confusions.

In [ ]:
from IPython.display import Image, display

fig_dir = "../figures"
for m in ["RNN", "LSTM", "FINBERT"]:
    path = os.path.join(fig_dir, f"confusion_matrix_{m.lower()}.png")
    if os.path.exists(path):
        print(f"=== {m} Confusion Matrix ===")
        display(Image(filename=path))
        print("\n")